# MLFlow Search Documentation

This notebook demonstrates how to use MLFlow to track asteroid search
experiments with `find-asteroids`, and how to list experiments and
print compiled results.

**Prerequisites:** Install the pipeline extras to get MLFlow support:

```bash
pip install "find-asteroids[pipeline]"
```

The notebook will:
1. Run a `find-asteroids` search under an MLFlow experiment, logging all
   search parameters via `mlflow.log_param`.
2. List all experiments in the MLFlow tracking store.
3. List the runs recorded under the experiment, printing their parameters
   and tags.
4. Compile the results across all runs and print a summary of each table.

## Setup

In [1]:
import tempfile
from pathlib import Path

import mlflow
from mlflow.tracking import MlflowClient

from find_asteroids.search import run_search_mlflow
from find_asteroids.results import compile_results_astropy

### Configuration

Set up data paths, the experiment name, the tracking URI, and search parameters.

Using a SQLite database (`"sqlite:///mlflow.db"`) as the tracking URI persists
experiment data between Python sessions.  Set `tracking_uri = None` to use the
default local `mlruns/` directory instead.

In [2]:
# Data files bundled with this repository (relative to docs/).
catalog = Path("catalog.ecsv")
psfs    = Path("psfs.ecsv")

# MLFlow experiment name.  Change this to group related runs together.
experiment = "asteroid-search-example"

# MLFlow tracking URI.  Using a SQLite database persists experiment data
# between Python sessions.  Set to None to use the default local
# mlruns/ directory instead.
tracking_uri = "sqlite:///mlflow.db"

# Search parameters
velocity    = [0.1, 0.5]   # deg / day  [min, max]
angle       = [0, 359.99]  # deg        [min, max]
dx          = 10            # bin-width in PSF units (i.e. 10 × median PSF width)
num_results = 10

## 1. Run a Search and Record it with MLFlow

Search parameters are passed as **keyword arguments** so that `run_search_mlflow`
automatically logs each one via `mlflow.log_param`.  The `results_dir` is created
inside a temporary directory; the results are uploaded to the tracking store as
artifacts, so the temporary directory can safely be removed after the run.

In [3]:
with tempfile.TemporaryDirectory() as tmpdir:
    results_dir = Path(tmpdir) / "results"

    run_id = run_search_mlflow(
        experiment,
        tracking_uri=tracking_uri,
        tags=[("dataset", "example-catalog")],
        catalog=str(catalog),
        psfs=str(psfs),
        velocity=velocity,
        angle=angle,
        dx=dx,
        num_results=num_results,
        results_dir=results_dir,
    )

print(f"MLFlow run ID: {run_id}")

2026/03/11 21:36:51 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/11 21:36:51 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


cluster has value 104 at (1853, 153, 70)
8067 / 8171 points remain
cluster has value 104 at (2867, 144, 87)
7963 / 8171 points remain
cluster has value 104 at (3015, 102, 66)
7859 / 8171 points remain
cluster has value 103 at (2601, 85, 64)
7756 / 8171 points remain
cluster has value 103 at (2666, 65, 44)
7653 / 8171 points remain
cluster has value 97 at (2274, 72, 58)
7556 / 8171 points remain
cluster has value 95 at (2733, 171, 43)
7461 / 8171 points remain
cluster has value 92 at (3247, 57, 85)
7369 / 8171 points remain
cluster has value 88 at (2732, 56, 81)
7281 / 8171 points remain
cluster has value 72 at (2795, 179, 67)
7209 / 8171 points remain
MLFlow run ID: 340b7d414a354623ab053f1ca5adf32d


## 2. List All Experiments

Use `MlflowClient.search_experiments()` to enumerate every experiment
in the tracking store.

In [4]:
mlflow.set_tracking_uri(tracking_uri)
client = MlflowClient()
experiments = client.search_experiments()

print("Experiments:")
for exp in experiments:
    print(
        f"  id={exp.experiment_id!r:>4}  "
        f"name={exp.name!r}  "
        f"artifact_location={exp.artifact_location!r}"
    )

Experiments:
  id= '1'  name='asteroid-search-example'  artifact_location='/Users/steven/Projects/find_asteroids/docs/notebooks/mlruns/1'
  id= '0'  name='Default'  artifact_location='/Users/steven/Projects/find_asteroids/docs/notebooks/mlruns/0'


## 3. Inspect Runs in an Experiment

Use `MlflowClient.search_runs()` to list all runs under the experiment.
Each run shows its run ID, status, logged parameters, and custom tags.

In [5]:
exp = client.get_experiment_by_name(experiment)
runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="",
    max_results=5000,
)

print(f"Runs in experiment '{experiment}' ({len(runs)} total):")
for run in runs:
    # Omit internal mlflow.* tag keys
    user_tags = {
        k: v
        for k, v in run.data.tags.items()
        if not k.startswith("mlflow.")
    }
    print(
        f"\n  run_id : {run.info.run_id}\n"
        f"  status : {run.info.status}\n"
        f"  params : {run.data.params}\n"
        f"  tags   : {user_tags}"
    )

Runs in experiment 'asteroid-search-example' (2 total):

  run_id : 340b7d414a354623ab053f1ca5adf32d
  status : FINISHED
  params : {'catalog': 'catalog.ecsv', 'psfs': 'psfs.ecsv', 'velocity_1': '0.1', 'velocity_2': '0.5', 'angle_1': '0', 'angle_2': '359.99', 'dx': '10', 'num_results': '10', 'results_dir': '/var/folders/2h/8z6jx4_d0n9148pwtsl22sqw0000gn/T/tmpcpoc_4s7/results'}
  tags   : {'dataset': 'example-catalog', 'versions_find_asteroids': '0.1.4.dev8'}

  run_id : abe562c15e8547ea8e9a23bf3ae157d0
  status : FINISHED
  params : {'catalog': 'catalog.ecsv', 'psfs': 'psfs.ecsv', 'velocity_1': '0.1', 'velocity_2': '0.5', 'angle_1': '0', 'angle_2': '359.99', 'dx': '10', 'num_results': '10', 'results_dir': '/var/folders/2h/8z6jx4_d0n9148pwtsl22sqw0000gn/T/tmpmlkqj39n/results'}
  tags   : {'dataset': 'example-catalog', 'versions_find_asteroids': '0.1.4.dev8'}


## 4. Compile and Print Results

Use `compile_results_astropy` with `reader="mlflow"` to download artifacts
from all runs in the experiment and stack them into four Astropy tables:

| Table | Contents |
|-------|----------|
| `result` | Hough-space peak: x, y, direction, vote count *n* |
| `tracklet` | Refined on-sky trajectory: velocities, positions, uncertainties |
| `points` | Catalog detections that voted for each result |
| `gathered` | Original catalog entries matched to each result |

> **Note:** The tracking URI must be set via `mlflow.set_tracking_uri()` before
> calling `compile_results_astropy` so that the underlying `read_results_mlflow`
> uses the correct tracking store.

In [6]:
# Set the tracking URI before compiling so the correct store is used.
mlflow.set_tracking_uri(tracking_uri)

for name, table in compile_results_astropy(
    experiment, reader="mlflow", output_format="ecsv"
):
    print(f"\n-- Table: '{name}' --")
    print(f"   rows    : {len(table)}")
    print(f"   columns : {table.colnames}")
    table.pprint(max_lines=5, max_width=200)


-- Table: 'gathered' --
   rows    : 2016
   columns : ['ra', 'dec', 'sigma_x', 'flux', 'flux_sigma', 'significance', 'time', 'exposures', 'detectors', 'peakValue', 'psi', 'phi', 'i_x', 'i_y', 'result_num', 'run_id', 'run_tags', 'run_params', 'run_hash']
        ra                 dec         ...             run_hash            
------------------ ------------------- ... --------------------------------
216.23985376668162 -11.057847830145272 ... 9727bf7ac487b0afe94010de48d7f4f3
216.23928525307826 -11.057995462681339 ... 9727bf7ac487b0afe94010de48d7f4f3
               ...                 ... ...                              ...
216.26180853038946 -11.050962447314031 ... 9727bf7ac487b0afe94010de48d7f4f3
 216.2613676832834 -11.050854187284886 ... 9727bf7ac487b0afe94010de48d7f4f3
Length = 2016 rows

-- Table: 'result' --
   rows    : 20
   columns : ['x', 'y', 'direction', 'n', 'n1', 'n2', 'n5', 'n10', 'result_num', 'run_id', 'run_tags', 'run_params', 'run_hash']
 x    y  ...             